## Document Forgery Detection: Machine Learning Classification Report

### Overview

This notebook trains and evaluates a Random Forest model for forged-document detection using features produced by `dataset_builder.py / feature_engineering.py`.
The report emphasizes stakeholder-ready evidence for banking use cases:
- **Accuracy and class-level quality** (precision, recall, F1-score)
- **Confusion matrix** on on both hold-out and generated train/validation/testing data
- **Risk-oriented metrics** like false positive/negative rates
- **Model explainability** via top forensic indicators


In [ ]:
import json
import os
from pathlib import Path
import joblib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Load datasets generated from `training_set` / `validation_set` / `testing_set`
TRAIN_DATASET_PATH = "dataset_outputs/train.csv"
VALIDATION_DATASET_PATH = "dataset_outputs/val.csv"
TEST_DATASET_PATH = "dataset_outputs/test.csv"

if not Path(TRAIN_DATASET_PATH).exists():
    raise FileNotFoundError(f"Missing required dataset: {TRAIN_DATASET_PATH}")

assert Path(VALIDATION_DATASET_PATH).exists(), "Validation dataset is required."
assert Path(TEST_DATASET_PATH).exists(), "Test dataset is required."

df_train_raw = pd.read_csv(TRAIN_DATASET_PATH)
df_validation = pd.read_csv(VALIDATION_DATASET_PATH)
df_test = pd.read_csv(TEST_DATASET_PATH)

print("Training dataset shape:", df_train_raw.shape)
print("Validation dataset shape:", df_validation.shape)
print("Test dataset shape:", df_test.shape)

display(df_train_raw.head())

print("Training label distribution:")
print(df_train_raw["Label"].value_counts())
assert df_train_raw["Label"].nunique() == 2, (
    "Dataset contains only one class; forgery detection requires genuine + forged examples."
)

print("Validation label distribution:")
print(df_validation["Label"].value_counts())

print("Test label distribution:")
print(df_test["Label"].value_counts())


In [ ]:
# =========================
# STEP 1: Split Features & Labels
# =========================

# Training set
X_train = df_train_raw.drop(columns=["Label"])
y_train = df_train_raw["Label"]

# Validation set
assert df_validation is not None, "Validation dataset is required"
X_val = df_validation.drop(columns=["Label"])
y_val = df_validation["Label"]

# Test set
assert df_test is not None, "Test dataset is required"
X_test = df_test.drop(columns=["Label"])
y_test = df_test["Label"]

print("Dataset Shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


In [ ]:
# Feature / Target prep + leakage cleanup (NO manual one-hot encoding)
LEAKAGE_COLUMNS = ["Document_ID", "Image_Name"]

def drop_leakage_columns(df):
    leakage_prefixes = ("Image_Name_", "Country_Code_", "Country_Name_")
    drop_cols = [
        col for col in df.columns
        if col in LEAKAGE_COLUMNS or any(col.startswith(prefix) for prefix in leakage_prefixes)
    ]
    if drop_cols:
        print("[WARN] Dropping leakage/pre-encoded columns:", sorted(drop_cols))
    return df.drop(columns=drop_cols, errors="ignore")

X_train = drop_leakage_columns(X_train)
X_val = drop_leakage_columns(X_val)
X_test = drop_leakage_columns(X_test)

y_train = y_train.astype(int)
y_val = y_val.astype(int)
y_test = y_test.astype(int)

categorical_features = [c for c in ["Country_Code", "Country_Name"] if c in X_train.columns]
numerical_features = [c for c in X_train.columns if c not in categorical_features]

print("Categorical features:", categorical_features)
print("Numerical feature count:", len(numerical_features))
print("Final train feature count:", X_train.shape[1])


### Train/Test Split (70:30 Ratio)

In [ ]:
# Dataset split summary (predefined train/validation/test files)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))


### Baseline Model (Majority Class Predictor)


In [ ]:
# =========================
# BASELINE MODEL (Majority Class)
# =========================

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

# Baseline model: predicts most frequent class
baseline_model = DummyClassifier(strategy="most_frequent")

# Fit ONLY on training labels (no real learning)
baseline_model.fit(X_train, y_train)

print("[INFO] Baseline model (Majority Class) initialized.")


In [ ]:
# =========================
# BASELINE - VALIDATION PERFORMANCE
# =========================

val_pred_baseline = baseline_model.predict(X_val)

val_acc_baseline = accuracy_score(y_val, val_pred_baseline)

print("\n[BASELINE VALIDATION RESULTS]")
print(f"Accuracy: {val_acc_baseline:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, val_pred_baseline))


In [ ]:
# =========================
# BASELINE - TEST PERFORMANCE
# =========================

test_pred_baseline = baseline_model.predict(X_test)

test_acc_baseline = accuracy_score(y_test, test_pred_baseline)

print("\n[BASELINE TEST RESULTS]")
print(f"Accuracy: {test_acc_baseline:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_pred_baseline))


### Random Forest Classifier Training

In [ ]:
# =========================
# STEP 2: Train Full Pipeline (Preprocessing + RandomForest)
# =========================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numerical_features,
        ),
    ],
    remainder="drop",
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
        ),
    ),
])

model.fit(X_train, y_train)

print("[INFO] Full sklearn pipeline trained (preprocessing + classifier).")


In [ ]:
# =========================
# STEP 3: Validation Evaluation
# =========================
from sklearn import metrics

def evaluate_classifier(model, X_eval, y_eval, dataset_name="Dataset"):
    preds = model.predict(X_eval)
    probs = model.predict_proba(X_eval)[:, 1] if hasattr(model, "predict_proba") else None

    precision, recall, f1, _ = precision_recall_fscore_support(
       y_eval,
       preds,
       average='binary',
       zero_division=0,
       )

    metrics = {
        "dataset": dataset_name,
        "accuracy": float(accuracy_score(y_eval, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_eval, preds)),
        "precision" : float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "mcc": float(matthews_corrcoef(y_eval, preds)),
        "roc_auc": float(roc_auc_score(y_eval, probs)) if probs is not None else None,
        "classification_report" : classification_report(
            y_eval,
            preds,
            target_names=["Genuine", "Forged"],
            output_dict=True,
            zero_division=0,
            )
     }

    tn, fp, fn, tp = confusion_matrix(y_eval, preds).ravel()
    metrics.update(
        {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) > 0 else 0.0,
            "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) > 0 else 0.0,
        }
    )

    print(f"=== RANDOM FOREST RESULTS ({dataset_name}) ===")
    print(f"Accuracy:           {metrics['accuracy']:.4f}")
    print(f"Balanced Accuracy:  {metrics['balanced_accuracy']:.4f}")
    print(f"Precision:          {metrics['precision']:.4f}")
    print(f"Recall:             {metrics['recall']:.4f}")
    print(f"F1-Score:           {metrics['f1_score']:.4f}")
    print(f"MCC:                {metrics['mcc']:.4f}")
    if metrics["roc_auc"] is not None:
        print(f"ROC AUC:        {metrics['roc_auc']:.4f}")
    print(f"FPR / FNR:         {metrics['false_positive_rate']:.4f} / {metrics['false_negative_rate']:.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    print(classification_report(y_eval, preds, target_names=["Genuine", "Forged"], zero_division=0))

    return preds, metrics

val_pred = model.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)

_, validation_metrics = evaluate_classifier(model, X_val, y_val, dataset_name="Validation Set")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_val, val_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds",
            xticklabels=["Authentic", "Forged"],
            yticklabels=["Authentic", "Forged"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Validation Set)")
plt.show()


#### Save the Trained Model

This avoids having to retrain the model every time you want to make a prediction.

In [ ]:
MODEL_DIR = "trained_models"
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "forged_document_rf_model.pkl")
joblib.dump(model, model_path)
print(f"Trained model saved to: {model_path}")


In [ ]:
metrics_path = os.path.join(MODEL_DIR, "training_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({"validation_set": validation_metrics}, f, indent=2)
print(f"Validation metrics saved to: {metrics_path}")


### Key Results and Risk Signals for Banking Stakeholders

The next sections provide model evidence relevant to financial institutions:
- How many genuine documents may be wrongly blocked (**False Positives**)
- How many forged documents may pass undetected (**False Negatives**)
- Which forensic signals contribute most to automated screening decisions

### Confusion Matrix (Hold-out Test Set)

The Confusion Matrix visually represents the model's performance on the testing set, detailing true positives, true negatives, and classification errors.

In [ ]:
# =========================
# STEP 4: Final Test Evaluation
# =========================

from sklearn.metrics import confusion_matrix

test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)

_, testing_metrics = evaluate_classifier(model, X_test, y_test, dataset_name="Test Set")

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({"validation_set": validation_metrics, "testing_set": testing_metrics}, f, indent=2)

print(f"Updated metrics saved to: {metrics_path}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=["Authentic", "Forged"],
            yticklabels=["Authentic", "Forged"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Test Set)")
plt.show()


In [ ]:
# =========================
# MODEL COMPARISON
# =========================

print("\n==============================")
print(" BASELINE vs TRAINED MODEL")
print("==============================")

print(f"Baseline Test Accuracy : {test_acc_baseline:.4f}")
print(f"Trained Model Accuracy : {test_acc:.4f}")

improvement = test_acc - test_acc_baseline

print(f"\nImprovement: {improvement:.4f} ({improvement*100:.2f}%)")


In [ ]:
import matplotlib.pyplot as plt

models = ["Baseline", "RandomForest"]
accuracies = [test_acc_baseline, test_acc]

plt.figure(figsize=(5,5))
plt.bar(models, accuracies, color= "Pink")

plt.ylabel("Accuracy")
plt.title("Model Comparison")

for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.2f}", ha="center")

plt.show()


#### Feature Importance

This chart highlights the most influential forensic features used by the Random Forest model. These indicators can guide compliance teams when prioritizing manual review and fraud controls.

In [ ]:
# Feature importances from trained pipeline
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(
    classifier.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

importances.head(15)


In [ ]:
results = X_test.copy()
results["True_Label"] = y_test.values
results["Predicted_Label"] = test_pred

errors = results[results["True_Label"] != results["Predicted_Label"]]
print("Misclassified test samples:", len(errors))

errors.head()


In [ ]:
# Top 10 forensic indicator
top_features = importances.nlargest(10)
plt.figure(figsize=(10, 6))
top_features = importances.nlargest(10)
sns.barplot(x=top_features.values, y=top_features.index, palette="viridis")
plt.title('Top 10 Forensic Indicators for Forgery Detection')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

In [ ]:
# Executive-ready summary table
summary_rows = [
    {
        "Dataset": "Validation Set",
        "Accuracy": validation_metrics["accuracy"],
        "Balanced Accuracy": validation_metrics["balanced_accuracy"],
        "Precision": validation_metrics["precision"],
        "Recall": validation_metrics["recall"],
        "F1-Score": validation_metrics["f1_score"],
        "False Positive Rate": validation_metrics["false_positive_rate"],
        "False Negative Rate": validation_metrics["false_negative_rate"],
    },
    {
        "Dataset": "Testing Set",
        "Accuracy": testing_metrics["accuracy"],
        "Balanced Accuracy": testing_metrics["balanced_accuracy"],
        "Precision": testing_metrics["precision"],
        "Recall": testing_metrics["recall"],
        "F1-Score": testing_metrics["f1_score"],
        "False Positive Rate": testing_metrics["false_positive_rate"],
        "False Negative Rate": testing_metrics["false_negative_rate"],
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_df.style.format({
        "Accuracy": "{:.4f}",
        "Balanced Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}",
        "False Positive Rate": "{:.4f}",
        "False Negative Rate": "{:.4f}",
})


In [ ]:
import pickle

MODEL_PATH = "trained_models/forged_document_rf_model.pkl"

with open(MODEL_PATH, "wb") as f:
    pickle.dump(model, f)

print(f"[INFO] Model saved to {MODEL_PATH}")
